# VQ-VAE-2 Latents — Code Replacement (CN vs AD)

Swap codebook entries and decode to compare structural effects on healthy vs diseased subjects. NIfTI export of originals, recons, and diffs.

In [ ]:
import sys
import os

# Add project root to sys.path (this notebook lives in eval/)
sys.path.insert(0, os.path.abspath(".."))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import json

# Import local modules
import models.vqvae as vqvae
import utils.utils as utils
from utils.utils import load_data, CreateBrainMaskd, ApplyBrainMaskd

# ============================================================
# CONFIGURATION
# ============================================================
# Use vqvae_best.pt for the best checkpoint (by rolling avg total loss),
# or vqvae_model.pt for the latest checkpoint.
CHECKPOINT_PATH = (
    "/home/ng24/projects/multiview-crl/results/ADNI_registered/multiview-06-content-lr-002-single-level/vqvae_best1.pt"
)
DATA_DIR = "/data/natalia/ADNI_stripped"
CSV_PATH = "/home/ng24/projects/nmpevqvae/labels_cleaned_3class.csv"
NUM_SAMPLES = 200  # number of subjects to process
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ADNI 2-view split (must match training)
CONTENT_SIZE = 256  # dimensions treated as content (shared between T1/T2)
STYLE_SIZE = 256  # dimensions treated as style (view-specific)

# Resampling: "original" (1 mm, ~182x218x182) or "2mm" (91x109x91)
RESAMPLE_MODE = "original"
# ============================================================

print(f"Using device: {DEVICE}")

## 1. Load checkpoint & build model

In [ ]:
# Load settings saved alongside the checkpoint
settings_path = os.path.join(os.path.dirname(CHECKPOINT_PATH), "settings.json")
with open(settings_path, "r") as f:
    settings = json.load(f)

print("Model settings:")
for k in [
    "vqvae_hidden_channels",
    "vqvae_res_channels",
    "vqvae_nb_res_layers",
    "vqvae_nb_levels",
    "vqvae_embed_dim",
    "vqvae_nb_entries",
    "vqvae_scaling_rates",
]:
    print(f"  {k}: {settings.get(k, 'N/A')}")

# Override CONTENT_SIZE / STYLE_SIZE from saved settings.
# settings.json stores content_indices (list[list[int]]) and style_indices (list[int])
# written by update_args() at training time — use them so the VQVAE is rebuilt
# with exactly the same content/style ratio that was used during training.
# Falling back to the hardcoded values only when the keys are absent (old checkpoints).
if "content_indices" in settings and "style_indices" in settings:
    CONTENT_SIZE = len(settings["content_indices"][0])
    STYLE_SIZE = len(settings["style_indices"])
    print(f"\n  content_size : {CONTENT_SIZE}  (from settings.json['content_indices'])")
    print(f"  style_size   : {STYLE_SIZE}  (from settings.json['style_indices'])")
else:
    print(f"\n  ⚠ content_indices / style_indices not found in settings.json.")
    print(f"    Using hardcoded CONTENT_SIZE={CONTENT_SIZE}, STYLE_SIZE={STYLE_SIZE}.")
    print(f"    If these don't match the training run, content_channels will be wrong.")

In [ ]:
# Build model — must match the exact architecture used during training.
# inject_style_to_decoder controls whether decoders[0] has a final_conv branch;
# if it differs from the checkpoint the decoder weights will silently not load.
inject_style = settings.get("inject_style_to_decoder", False)

# ── Auto-detect content/style split from checkpoint weights ──────────────────
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
state_dict = checkpoint["encoders"]

# Strip MoCo "online." prefix if present to find the right keys
_prefix = ""
if any(k.startswith("online.") for k in state_dict):
    _prefix = "online."

# channel_logits is now a ParameterDict keyed by level.
# Old checkpoints: "module.channel_logits" (single param)
# New checkpoints: "module.channel_logits.0", "module.channel_logits.1", etc.
_cl_key_old = f"{_prefix}module.channel_logits"

# Find all channel_logits.X keys where X is a digit
_cl_levels = []
for k in state_dict:
    stripped = k[len(_prefix) :] if k.startswith(_prefix) else k
    if stripped.startswith("module.channel_logits."):
        suffix = stripped[len("module.channel_logits.") :]
        if suffix.isdigit():
            _cl_levels.append(int(suffix))
_cl_levels = sorted(_cl_levels)

# ── Detect mask_mode from checkpoint keys ────────────────────────────────────
# fixed mode: has fixed_mask_* buffers, no channel_logits
# learned mode: has channel_logits params
# onthefly mode: neither (logits computed from activations at runtime)
_has_fixed_mask = any(
    (k[len(_prefix) :] if k.startswith(_prefix) else k).startswith("module.fixed_mask_") for k in state_dict
)

if _has_fixed_mask:
    _detected_mask_mode = "fixed"
elif _cl_levels or _cl_key_old in state_dict:
    _detected_mask_mode = "learned"
else:
    _detected_mask_mode = "onthefly"

# Allow settings.json to override (it's always authoritative if present)
_mask_mode = settings.get("mask_mode", _detected_mask_mode)

if _cl_levels:
    # New-style per-level channel_logits
    _cl_key_0 = f"{_prefix}module.channel_logits.{_cl_levels[0]}"
    _mask_dim = state_dict[_cl_key_0].shape[0]
    content_style_levels = _cl_levels
elif _cl_key_old in state_dict:
    # Old-style single channel_logits
    _mask_dim = state_dict[_cl_key_old].shape[0]
    content_style_levels = [0]
else:
    _mask_dim = None
    # No channel_logits (onthefly or fixed mode) — read content_style_levels from settings
    content_style_levels = settings.get("content_style_levels", [0])

# ── Per-level content_channels detection from codebook input sizes ────────────
hidden_channels = settings["vqvae_hidden_channels"]
_embed_dim = settings["vqvae_embed_dim"]
_nb_levels = settings["vqvae_nb_levels"]

# Detect per-level content_channels from codebook conv_in dimensions.
# Works for any mask_mode (learned, onthefly, or fixed) as long as the codebook was
# sized for the content subset (content_channels < hidden_channels).
_content_ch_per_level = {}
_used_codebook_detection = False
content_ratios = None

for lvl in content_style_levels:
    _cb_key = f"{_prefix}module.codebooks.{lvl}.conv_in.weight"
    if _cb_key in state_dict:
        _cb_in = state_dict[_cb_key].shape[1]
        if lvl == _nb_levels - 1:
            # Top level: codebook input = content_channels (no embed_dim concat)
            _content_ch_per_level[lvl] = _cb_in
        else:
            # Lower levels: codebook input = content_channels + embed_dim
            _content_ch_per_level[lvl] = _cb_in - _embed_dim

# Check if codebook detection is valid: if ALL detected levels give
# content_channels == hidden_channels, the checkpoint predates per-level
# codebook sizing — fall back to settings-based CONTENT_SIZE / STYLE_SIZE.
_all_full_width = all(cc == hidden_channels for cc in _content_ch_per_level.values()) if _content_ch_per_level else True

if not _all_full_width:
    # Codebook was sized for content masking — use detected values
    _used_codebook_detection = True

    _first_lvl = content_style_levels[0]
    _content_channels = _content_ch_per_level.get(_first_lvl, hidden_channels)
    _style_channels = hidden_channels - _content_channels

    CONTENT_SIZE = _content_channels
    STYLE_SIZE = _style_channels

    # Compute per-level content_ratios if levels have different content sizes
    _ratios = [_content_ch_per_level[lvl] / hidden_channels for lvl in content_style_levels]
    if len(set(_ratios)) > 1:
        content_ratios = _ratios

    _mode_desc = {
        "learned": "learned",
        "onthefly": "onthefly (no channel_logits)",
        "fixed": "fixed (first K = content)",
    }
    print(f"Auto-detected from checkpoint (per-level codebook sizing):")
    print(f"  mask_mode             : {_mode_desc.get(_mask_mode, _mask_mode)}")
    print(f"  content_style_levels  : {content_style_levels}")
    for lvl in content_style_levels:
        cc = _content_ch_per_level.get(lvl, "?")
        print(f"  level {lvl} content_ch    : {cc} / {hidden_channels}  ({cc/hidden_channels:.1%})")
    print(f"  -> CONTENT_SIZE={CONTENT_SIZE}, STYLE_SIZE={STYLE_SIZE}")
    if content_ratios:
        print(f"  -> content_ratios={content_ratios}")
else:
    # Old checkpoint: codebooks use full hidden_channels.
    # Keep CONTENT_SIZE / STYLE_SIZE from settings cell (cell above).
    print(f"Codebook detection gives content_ch == hidden_channels at all levels.")
    print(f"  content_style_levels  : {content_style_levels}")
    print(f"  → Using CONTENT_SIZE={CONTENT_SIZE}, STYLE_SIZE={STYLE_SIZE} from settings.json")

# ── Detect whether checkpoint uses hidden_channels or embed_dim masking ──────
if _mask_dim is not None and _mask_dim != hidden_channels:
    raise RuntimeError(
        f"Checkpoint has channel_logits of size {_mask_dim} but hidden_channels={hidden_channels}. "
        f"This checkpoint was likely trained with an older code version where the mask was on "
        f"embed_dim={_embed_dim}. You need to checkout the matching code version "
        f"to load this checkpoint."
    )

# ── Detect separate_encoders from checkpoint keys ────────────────────────────
# If the checkpoint contains "module.encoders_v1.*" keys, the model was trained
# with --separate-encoders.
_sep_enc = any(
    (k[len(_prefix) :] if k.startswith(_prefix) else k).startswith("module.encoders_v1.") for k in state_dict
)
if _sep_enc:
    print("Detected separate_encoders in checkpoint (encoders_v1 keys found).")

_style_inj_mode = settings.get("style_injection_mode", "concat")

# ── Read training flags that affect the encoder forward pass ─────────────────
# These MUST match training exactly, otherwise the encoder produces different
# features at inference time (e.g. style channels zeroed between levels when
# they should be passed through, causing out-of-distribution inputs at higher levels).
_pass_full = settings.get("pass_full_to_next_level", False)
_narrow_enc = settings.get("narrow_encoder_input", False)
_top_recon = settings.get("top_level_recon_only", False)
_quantize_style = settings.get("quantize_style", False)
_style_embed_dim = settings.get("style_embed_dim", None)
_style_nb_entries_raw = settings.get("style_nb_entries", None)
_use_content_proj = settings.get("use_content_projection", False)

# ── Normalize per-level codebook sizes ───────────────────────────────────────
# --vqvae-nb-entries / --style-nb-entries are argparse nargs="+" → always a
# list in settings.json. W&B sweeps occasionally round-trip these as
# stringified lists ("[384]") or nested lists ([[384]]), which then reach
# CodeLayer where torch.randn(embed_dim, nb_entries) raises
#   TypeError: randn(): argument 'size' failed to unpack ... got list
# Flatten + coerce to int / flat list[int] before handing to VQVAE.
import ast as _ast


def _normalize_entries(v):
    if v is None:
        return None
    if isinstance(v, str):
        try:
            v = _ast.literal_eval(v)
        except (ValueError, SyntaxError):
            return int(v)
    if isinstance(v, (list, tuple)):
        flat = []
        for x in v:
            if isinstance(x, (list, tuple)):
                flat.extend(x)
            else:
                flat.append(x)
        flat = [int(x) for x in flat]
        return flat[0] if len(flat) == 1 else flat
    return int(v)


_nb_entries = _normalize_entries(settings["vqvae_nb_entries"])
_style_nb_entries = _normalize_entries(_style_nb_entries_raw)
print(f"vqvae_nb_entries (normalized) : {_nb_entries}")
if _style_nb_entries is not None:
    print(f"style_nb_entries (normalized) : {_style_nb_entries}")

# ── Optional: skip levels in the final reconstruction decoder ────────────────
# Define SKIP_RECON_LEVELS in a cell above to ablate finer codes, e.g.
#     SKIP_RECON_LEVELS = [0, 1]
# zeros out levels 0 and 1 in the final (level-0) decoder concat, leaving only
# the top (coarsest) codes to drive reconstruction. The top level itself cannot
# be skipped. If undefined, falls back to whatever was stored at training time.
try:
    _skip_recon = SKIP_RECON_LEVELS  # noqa: F821 — optional user override
    print(f"⚠ Notebook override: skip_decoder_concat_levels={_skip_recon}")
except NameError:
    _skip_recon = settings.get("skip_decoder_concat_levels", None)
    if _skip_recon:
        print(f"skip_decoder_concat_levels (from settings): {_skip_recon}")

vqvae_model = vqvae.VQVAE(
    in_channels=1,
    hidden_channels=hidden_channels,
    res_channels=settings["vqvae_res_channels"],
    nb_res_layers=settings.get("vqvae_nb_res_layers", 2),
    nb_levels=_nb_levels,
    embed_dim=_embed_dim,
    nb_entries=_nb_entries,
    scaling_rates=settings["vqvae_scaling_rates"],
    content_size=CONTENT_SIZE,
    style_size=STYLE_SIZE,
    inject_style_to_decoder=inject_style,
    content_style_levels=content_style_levels,
    content_ratios=content_ratios,
    separate_encoders=_sep_enc,
    mask_mode=_mask_mode,
    style_injection_mode=_style_inj_mode,
    pass_full_to_next_level=_pass_full,
    narrow_encoder_input=_narrow_enc,
    top_level_recon_only=_top_recon,
    quantize_style=_quantize_style,
    style_embed_dim=_style_embed_dim,
    style_nb_entries=_style_nb_entries,
    use_content_projection=_use_content_proj,
    skip_decoder_concat_levels=_skip_recon,
)

content_channels = vqvae_model.content_channels
print(f"\nhidden_channels         : {hidden_channels}")
print(f"content_channels        : {content_channels}")
if vqvae_model.content_channels_per_level:
    print(f"content_channels_per_lvl: {vqvae_model.content_channels_per_level}")
print(f"content_style_levels    : {content_style_levels}")
print(f"mask_mode               : {_mask_mode}")
print(f"inject_style_to_decoder : {inject_style}")
print(f"separate_encoders       : {_sep_enc}")
print(f"style_injection_mode    : {_style_inj_mode}")
print(f"pass_full_to_next_level : {_pass_full}")
print(f"narrow_encoder_input    : {_narrow_enc}")
print(f"top_level_recon_only    : {_top_recon}")
print(f"quantize_style          : {_quantize_style}")
print(f"use_content_projection  : {_use_content_proj}")

# Wrap in DataParallel to match the saved checkpoint structure
vqvae_model = torch.nn.DataParallel(vqvae_model, device_ids=[0])

# ── Key remapping ─────────────────────────────────────────────────────────────
new_state_dict = {}
is_moco_checkpoint = any(k.startswith("online.") for k in state_dict)
if is_moco_checkpoint:
    print("Detected MoCoEncoder checkpoint — stripping 'online.' prefix.")

for k, v in state_dict.items():
    k = k.replace(".blocks.", ".stack.")  # legacy rename
    # Migrate old single channel_logits → ParameterDict key "0"
    if k.endswith("module.channel_logits") and not any(c.isdigit() for c in k.split(".")[-1]):
        k = k + ".0"
    if k.startswith("online."):
        new_state_dict[k[len("online.") :]] = v
    elif k.startswith(
        ("momentum_encoders.", "momentum_codebook_projs.", "momentum_content_proj.", "momentum_level0_proj.", "queue_")
    ):
        pass  # MoCo-only state — discard
    elif k.startswith("momentum_encoders_v1."):
        pass  # MoCo momentum copy of view-1 encoder — discard
    else:
        new_state_dict[k] = v

# ── Drop keys with shape mismatches (e.g. projection head from older config) ─
_model_sd = vqvae_model.state_dict()
_shape_skipped = []
for k in list(new_state_dict):
    if k in _model_sd and new_state_dict[k].shape != _model_sd[k].shape:
        _shape_skipped.append(f"{k}: ckpt {new_state_dict[k].shape} vs model {_model_sd[k].shape}")
        del new_state_dict[k]
if _shape_skipped:
    print(f"\u26a0 Skipped {len(_shape_skipped)} keys with shape mismatch:")
    for s in _shape_skipped:
        print(f"    {s}")

missing, unexpected = vqvae_model.load_state_dict(new_state_dict, strict=False)

missing_real = [k for k in missing if "num_batches_tracked" not in k]
unexpected_real = [k for k in unexpected if "num_batches_tracked" not in k]

if missing_real:
    print(f"\u26a0 Missing keys ({len(missing_real)}): {missing_real[:6]}{'...' if len(missing_real) > 6 else ''}")
if unexpected_real:
    print(
        f"\u26a0 Unexpected keys ({len(unexpected_real)}): {unexpected_real[:6]}{'...' if len(unexpected_real) > 6 else ''}"
    )
if not missing_real and not unexpected_real:
    print("\u2713 Checkpoint loaded cleanly")
elif not missing_real:
    print("\u2713 Checkpoint loaded (unexpected keys are harmless extras in the file)")

vqvae_model.to(DEVICE)
vqvae_model.eval()
print(f"\u2713 Model on {DEVICE}, step {checkpoint['step']}")
print(f"  Total params: {sum(p.numel() for p in vqvae_model.parameters()):,}")

In [ ]:
# -- Diagnostic: isolate where the constant-output problem lives --
import torch

vqvae_module = vqvae_model.module if hasattr(vqvae_model, "module") else vqvae_model

# 1. Check encoder weight statistics (are weights non-trivial?)
print("=== Encoder weight check ===")
for name, p in list(vqvae_module.encoders[0].named_parameters())[:6]:
    print(
        f"  {name}: shape={tuple(p.shape)}  "
        f"mean={p.data.mean():.6f}  std={p.data.std():.6f}  "
        f"absmax={p.data.abs().max():.6f}"
    )

# 2. Check ReZero alpha values (if 0, residual blocks are identity)
print("\n=== ReZero alpha values ===")
for name, p in vqvae_module.encoders[0].named_parameters():
    if "alpha" in name:
        print(f"  {name}: {p.data.item():.6f}")

# 3. Test raw encoder (bypass VQVAE forward)
print("\n=== Raw encoder test ===")
with torch.no_grad():
    d1 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)
    d2 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)

    # Run through encoder level 0 directly
    out1 = vqvae_module.encoders[0](d1)
    out2 = vqvae_module.encoders[0](d2)

    print(f"  Encoder output shape: {out1.shape}")
    print(f"  out1 range: [{out1.min():.4f}, {out1.max():.4f}]  mean={out1.mean():.6f}")
    print(f"  out2 range: [{out2.min():.4f}, {out2.max():.4f}]  mean={out2.mean():.6f}")

    # Check spatial variation (different voxels should have different values)
    print(f"  out1 spatial std (per-channel): {out1[0].std(dim=[1, 2, 3]).mean():.6f}")

    # Pool and compare
    p1 = out1.mean(dim=[2, 3, 4])
    p2 = out2.mean(dim=[2, 3, 4])
    cos = torch.nn.functional.cosine_similarity(p1, p2).item()
    print(f"  Pooled cosine similarity: {cos:.6f}")
    if cos > 0.999:
        print("  WARNING: Raw encoder also produces constant output!")
    else:
        print("  OK: Raw encoder works - bug is in VQVAE.forward()")

# 4. Test layer by layer to find where signal dies
print("\n=== Layer-by-layer signal propagation ===")
with torch.no_grad():
    x1 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)
    x2 = torch.randn(1, 1, 32, 32, 32, device=DEVICE)
    for layer_idx, layer in enumerate(vqvae_module.encoders[0].layers):
        x1 = layer(x1)
        x2 = layer(x2)
        cos = torch.nn.functional.cosine_similarity(x1.flatten(1), x2.flatten(1)).item()
        print(
            f"  After layer {layer_idx} ({type(layer).__name__}): "
            f"shape={tuple(x1.shape)}  cos={cos:.6f}  "
            f"range=[{x1.min():.3f},{x1.max():.3f}]"
        )

## 2. Load dataset & transforms

In [ ]:
df = pd.read_csv(CSV_PATH)
label_values = sorted(df["Group"].unique())
label_map = {v: i for i, v in enumerate(label_values)}
label_names = {i: v for v, i in label_map.items()}
print(f"Labels: {label_map}")

items, missing = load_data(df, DATA_DIR, label_map)
print(f"Loaded {len(items)} samples, {len(missing)} missing files")
items = items[:NUM_SAMPLES]
print(f"Using first {len(items)} subjects")

In [ ]:
# Match the training val pipeline EXACTLY. Previously the notebook always
# applied CreateBrainMaskd (intensity > 0 threshold). Training with masks on
# disk skips that and loads the proper brain-mask .nii.gz directly via
# LoadImaged — a different mask → different NormalizeIntensityd stats →
# different features → different modality probe result. Fix: delegate to
# utils.utils.transforms and use masks_from_disk when the CSV items carry
# mask paths.
from utils.utils import transforms as get_transforms

# Read spacing and spatial_size from settings.json (saved at training time)
spacing = settings.get("image_spacing", 2.0)
crop_margin = settings.get("crop_margin", 0)

_saved_spatial = settings.get("spatial_size", None)
if _saved_spatial is not None:
    spatial_size = tuple(_saved_spatial)
elif spacing == 1.0:
    spatial_size = (182, 218, 182)
else:
    spatial_size = (91, 109, 91)

# Mirror data/datasets.py:1283 — masks_from_disk is True iff every loaded
# item carries both mask paths.
masks_from_disk = all(("mask_image" in it) and ("mask_z_image" in it) for it in items)

_, _val_transforms_raw = get_transforms(
    spacing=spacing,
    crop_margin=crop_margin,
    spatial_size=spatial_size,
    masks_from_disk=masks_from_disk,
    asymmetric_aug=False,
)

# The raw val transform expects:
#   - ``mask_t1``/``mask_t2`` paths when ``masks_from_disk=True`` (loaded by LoadImaged)
#   - a ``label`` key at the end (required by the final ToTensord in utils.utils.transforms)
# Existing notebook cells pass only ``image_t1``/``image_t2``. Wrap the
# transform to auto-inject mask paths and a placeholder label, keyed on the
# T1 image path.
_img_to_extras = {
    it["image"]: {
        "mask_t1": it.get("mask_image"),
        "mask_t2": it.get("mask_z_image"),
        "label": it.get("label", 0),
    }
    for it in items
}


def val_transforms(data_dict):
    d = dict(data_dict)
    extras = _img_to_extras.get(d.get("image_t1"), {})
    if masks_from_disk:
        if "mask_t1" not in d and extras.get("mask_t1") is not None:
            d["mask_t1"] = extras["mask_t1"]
        if "mask_t2" not in d and extras.get("mask_t2") is not None:
            d["mask_t2"] = extras["mask_t2"]
    if "label" not in d:
        d["label"] = extras.get("label", 0)
    return _val_transforms_raw(d)


print(f"Transforms ready — spacing={spacing}mm, spatial_size={spatial_size}, " f"masks_from_disk={masks_from_disk}")
if not masks_from_disk:
    print(
        "  (No disk masks found on items — using intensity-threshold CreateBrainMaskd.\n"
        "  If training ran with disk masks, the modality probe will differ from the training log.)"
    )

## 3. Feature extraction

We call `vqvae_model(img, pool_only=True, return_recon=False)` which returns:
- `encoder_features`: list of `(B, embed_dim)` pooled vectors, one per level
  - All levels are pooled from the codebook projection (embed_dim space)
  - When `channel_logits` is active, content/style split is in this embed_dim space
- `estimated_content_indices`: the Gumbel-selected embedding dim indices at level 0

For visualisation we split level-0 pooled features into **content** and **style** subsets
using `estimated_content_indices`.

In [ ]:
nb_levels = settings["vqvae_nb_levels"]

# Storage
all_features = {f"level_{i}": [] for i in range(nb_levels)}
all_labels = []
all_modalities = []
all_subjects = []
all_content_indices = {}  # dict mapping level -> view-0 (T1) content indices
all_content_indices_v1 = {}  # dict mapping level -> view-1 (T2) content indices (only set for per-view masks)

print(f"Extracting latent features from {len(items)} subjects (T1 + T2 each) ...")

vqvae_module = vqvae_model.module if hasattr(vqvae_model, "module") else vqvae_model

# Map modality name → view_idx for separate-encoder models.
# view 0 = T1 (first modality), view 1 = T2 (second modality).
_VIEW_IDX = {"T1": 0, "T2": 1}

# Detect per-view masks: separate encoders with channel_logits_v1
_has_per_view = (
    getattr(vqvae_module, "separate_encoders", False) and getattr(vqvae_module, "channel_logits_v1", None) is not None
)

for idx, item in enumerate(items):
    if idx % 20 == 0:
        print(f"  {idx}/{len(items)} ...")

    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for modality, key in [("T1", "image_t1"), ("T2", "image_t2")]:
        img = transformed[key].unsqueeze(0).to(DEVICE)  # (1, 1, D, H, W)

        with torch.no_grad():
            # Get soft_content_masks (7th returned element)
            _, _, enc_features, _, _, _, soft_masks, *_ = vqvae_model(
                img,
                return_recon=False,
                pool_only=True,
                view_idx=_VIEW_IDX.get(modality, 0),
            )

        # enc_features is a list of (1, hidden_channels) pooled tensors per level.
        for level_idx, pooled in enumerate(enc_features):
            all_features[f"level_{level_idx}"].append(pooled.squeeze(0).cpu().float().numpy())

        # Record the content indices per level.
        #  - Tuple masks: training-style n_views=2 with per-view masks (unused
        #    here since we run n_views=1, but handled for completeness).
        #  - Single-tensor mask: single-view forward.  When per-view learned
        #    masks are active the returned tensor is the view-specific mask
        #    (channel_logits_v1 when view_idx=1), so T2 must be stored in
        #    all_content_indices_v1 to preserve the per-view split.
        for lvl, mask in soft_masks.items():
            if isinstance(mask, tuple):
                mask_v0, mask_v1 = mask
                if modality == "T1" and lvl not in all_content_indices:
                    all_content_indices[lvl] = torch.where(mask_v0.bool())[-1].tolist()
                elif modality == "T2" and lvl not in all_content_indices_v1:
                    all_content_indices_v1[lvl] = torch.where(mask_v1.bool())[-1].tolist()
            else:
                if modality == "T1" and lvl not in all_content_indices:
                    all_content_indices[lvl] = torch.where(mask.bool())[-1].tolist()
                elif modality == "T2" and _has_per_view and lvl not in all_content_indices_v1:
                    all_content_indices_v1[lvl] = torch.where(mask.bool())[-1].tolist()

        all_labels.append(item["label"])
        all_modalities.append(modality)
        all_subjects.append(item.get("subject", idx))

# Convert to numpy
for key in all_features:
    all_features[key] = np.array(all_features[key])
all_labels = np.array(all_labels)
all_modalities = np.array(all_modalities)

print(f"\nDone. {len(all_labels)} samples total.")
for key, val in all_features.items():
    print(f"  {key}: {val.shape}")

# Derive content / style subsets from the hidden_channels features for each masked level.
all_style_indices = {}
all_style_indices_v1 = {}

for lvl in all_content_indices.keys():
    all_style_indices[lvl] = [i for i in range(hidden_channels) if i not in set(all_content_indices[lvl])]
    if _has_per_view and lvl in all_content_indices_v1:
        all_style_indices_v1[lvl] = [i for i in range(hidden_channels) if i not in set(all_content_indices_v1[lvl])]
    else:
        all_style_indices_v1[lvl] = all_style_indices[lvl]

if len(all_content_indices.get(0, [])) > 0:
    for lvl in all_content_indices.keys():
        print(f"\n--- Level {lvl} ---")
        print(f"  content indices v0 ({len(all_content_indices[lvl])} ch): {all_content_indices[lvl][:8]}...")
        print(f"  style indices   v0 ({len(all_style_indices[lvl])} ch):   {all_style_indices[lvl][:8]}...")
        if _has_per_view and lvl in all_content_indices_v1:
            print(f"  content indices v1 ({len(all_content_indices_v1[lvl])} ch): {all_content_indices_v1[lvl][:8]}...")
            print(f"  style indices   v1 ({len(all_style_indices_v1[lvl])} ch):   {all_style_indices_v1[lvl][:8]}...")

## 11. Codebook Analysis — Extract Indices

Run a separate forward pass with `return_recon=True` so that non-coarsest codebook
levels get proper decoder conditioning (without it they receive zero-padded input
and produce incorrect indices).

We collect:
- **Per-level codebook usage histograms** (normalized frequency of each entry per subject)
- **Raw codebook indices** for the code-replacement experiment later

In [ ]:
nb_entries = vqvae_module.codebooks[0].n_embed
print(f"Codebook: {nb_levels} levels, {nb_entries} entries each\n")

# Storage: index 0 = finest (level 0), index nb_levels-1 = coarsest
cb_histograms = [[] for _ in range(nb_levels)]  # per-subject normalized histograms
cb_labels = []  # diagnosis label per sample
cb_modalities = []  # "T1" or "T2"

# Per-class examples for code-replacement comparison (CN vs AD)
class_examples = {}

# Position-wise code frequency accumulators (T1 only).
# pos_counts[level] has shape (D_l, H_l, W_l, nb_entries) — lazily initialized.
pos_counts = [None for _ in range(nb_levels)]
n_t1_subjects = 0  # number of T1 subjects contributing to pos_counts

print(f"Extracting codebook indices from {len(items)} subjects (T1 + T2) ...")
print(f"Will keep one T1 example per class for code-replacement demo: {list(label_names.values())}")

for idx, item in enumerate(items):
    if idx % 20 == 0:
        print(f"  {idx}/{len(items)} ...")

    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for modality, key in [("T1", "image_t1"), ("T2", "image_t2")]:
        img = transformed[key].unsqueeze(0).to(DEVICE)
        _vi = 0 if modality == "T1" else 1

        with torch.no_grad():
            recon, diffs, _, _, _, id_outputs_raw, _, _ = vqvae_model(
                img,
                return_recon=True,
                pool_only=False,
                view_idx=_vi,
            )

        # id_outputs_raw is coarsest-first; reverse to finest-first
        id_outputs = id_outputs_raw[::-1]

        for lvl in range(nb_levels):
            hist = torch.bincount(id_outputs[lvl].reshape(-1), minlength=nb_entries)
            hist = hist.float() / hist.sum()
            cb_histograms[lvl].append(hist.cpu().numpy())

        cb_labels.append(item["label"])
        cb_modalities.append(modality)

        # Accumulate position-wise code counts (T1 only to avoid modality confound)
        if modality == "T1":
            n_t1_subjects += 1
            for lvl in range(nb_levels):
                ids_lvl = id_outputs[lvl].squeeze(0).cpu()  # (D_l, H_l, W_l)
                if pos_counts[lvl] is None:
                    spatial = ids_lvl.shape
                    pos_counts[lvl] = torch.zeros(*spatial, nb_entries, dtype=torch.long)
                # Scatter-add: for each position, increment the count for the observed code
                pos_counts[lvl].scatter_(-1, ids_lvl.unsqueeze(-1), 1)

        # Keep one T1 example per diagnosis class
        if modality == "T1":
            class_name = label_names[item["label"]]
            if class_name not in class_examples:
                t1_tensor = transformed[key]
                ex_affine = None
                if hasattr(t1_tensor, "affine"):
                    ex_affine = t1_tensor.affine.numpy()
                elif hasattr(t1_tensor, "meta") and "affine" in t1_tensor.meta:
                    ex_affine = t1_tensor.meta["affine"].numpy()

                class_examples[class_name] = {
                    "id_outputs": [id_outputs[lvl].clone() for lvl in range(nb_levels)],
                    "image": img.clone(),
                    "item": item,
                    "affine": ex_affine,
                }
                print(f"  ✓ Stored {class_name} example: subject {item.get('subject', idx)}")

# Stack
for lvl in range(nb_levels):
    cb_histograms[lvl] = np.array(cb_histograms[lvl])
cb_labels = np.array(cb_labels)
cb_modalities = np.array(cb_modalities)

# Backward compat
_fallback_class = list(class_examples.keys())[-1] if class_examples else None
last_id_outputs = class_examples[_fallback_class]["id_outputs"] if _fallback_class else None
last_image = class_examples[_fallback_class]["image"] if _fallback_class else None
last_item = class_examples[_fallback_class]["item"] if _fallback_class else None
last_affine = class_examples[_fallback_class]["affine"] if _fallback_class else None

print(f"\n✓ Codebook extraction complete ({cb_histograms[0].shape[0]} samples)")
print(f"  T1 subjects for position-wise analysis: {n_t1_subjects}")
for lvl in range(nb_levels):
    used = (cb_histograms[lvl].sum(axis=0) > 0).sum()
    sp = tuple(pos_counts[lvl].shape[:-1]) if pos_counts[lvl] is not None else "?"
    print(
        f"  Level {lvl}: histogram shape {cb_histograms[lvl].shape}, "
        f"codes used: {used}/{nb_entries}, code map spatial: {sp}"
    )
print(f"  Class examples stored: {list(class_examples.keys())}")

In [ ]:
# ============================================================
# §14b. Class-discriminative code selection
# ------------------------------------------------------------
# Pick OLD_CODE / NEW_CODE based on which codes' usage frequency
# differs most between CN and AD subjects. Uses the per-subject
# T1 histograms collected in §11. Sets TARGET_LEVEL, OLD_CODE,
# NEW_CODE in scope; §15 picks them up via try/except NameError.
# ============================================================
from scipy import stats as _scipy_stats

# Restrict to T1 to remove the modality confound from the histograms
_t1_mask = cb_modalities == "T1"
_lab_t1 = cb_labels[_t1_mask]

TARGET_LEVEL = nb_levels - 1  # coarsest by default; sweep later if useful
H = cb_histograms[TARGET_LEVEL][_t1_mask]  # (n_t1, nb_entries)

_idx_cn, _idx_ad = label_map["CN"], label_map["AD"]
freqs_cn = H[_lab_t1 == _idx_cn]
freqs_ad = H[_lab_t1 == _idx_ad]
print(f"CN: n={len(freqs_cn)} T1 subjects;  AD: n={len(freqs_ad)} T1 subjects")
print(f"Level {TARGET_LEVEL}, nb_entries={H.shape[1]}\n")

mean_cn, mean_ad = freqs_cn.mean(0), freqs_ad.mean(0)
delta = mean_cn - mean_ad  # >0 → CN-enriched, <0 → AD-enriched

pvals = np.full(H.shape[1], np.nan)
for c in range(H.shape[1]):
    if freqs_cn[:, c].sum() == 0 and freqs_ad[:, c].sum() == 0:
        continue
    try:
        pvals[c] = _scipy_stats.mannwhitneyu(freqs_cn[:, c], freqs_ad[:, c], alternative="two-sided").pvalue
    except ValueError:
        pass  # all-equal → undefined; leave NaN

_used = (mean_cn + mean_ad) > 0
_sig = _used & (pvals < 0.05)
_eligible = _sig if _sig.any() else _used
if not _sig.any():
    print("⚠ no codes reach p<0.05 — falling back to top |Δfreq| among used codes\n")

cn_rank = np.where(_eligible & (delta > 0))[0]
ad_rank = np.where(_eligible & (delta < 0))[0]
cn_rank = cn_rank[np.argsort(-delta[cn_rank])]
ad_rank = ad_rank[np.argsort(delta[ad_rank])]

print("Top CN-enriched codes:")
for c in cn_rank[:5]:
    print(f"  code {c:4d}  Δfreq={delta[c]:+.3e}  p={pvals[c]:.2e}  " f"μ_CN={mean_cn[c]:.3e}  μ_AD={mean_ad[c]:.3e}")
print("\nTop AD-enriched codes:")
for c in ad_rank[:5]:
    print(f"  code {c:4d}  Δfreq={delta[c]:+.3e}  p={pvals[c]:.2e}  " f"μ_CN={mean_cn[c]:.3e}  μ_AD={mean_ad[c]:.3e}")

assert len(cn_rank) and len(ad_rank), "no class-enriched codes at this level"
OLD_CODE = int(cn_rank[0])  # CN-enriched
NEW_CODE = int(ad_rank[0])  # AD-enriched
print(f"\n→ OLD_CODE={OLD_CODE} (CN-enriched), NEW_CODE={NEW_CODE} (AD-enriched), " f"TARGET_LEVEL={TARGET_LEVEL}")

## 15. Codebook Vector Replacement — CN vs AD Comparison

Replace a specific codebook entry at a chosen level with a different entry, decode
back to an image, and compare the effect on a **healthy (CN)** and **diseased (AD)**
subject side by side. This reveals whether different codes capture class-specific
structural information (e.g., atrophy patterns in AD).

The model's `decode_codes` method handles the hierarchical decoding and automatically
provides a zero-style placeholder when `inject_style_to_decoder` is active.

In [ ]:
import torch.nn.functional as F

# ── Configuration ────────────────────────────────────────────────
# TARGET_LEVEL / OLD_CODE / NEW_CODE may be set upstream (e.g. by
# §14b class-discriminative selection). Honour those if defined,
# otherwise fall back to the single-subject heuristic.
try:
    TARGET_LEVEL
except NameError:
    TARGET_LEVEL = nb_levels - 1
SLICE_AXIS = 2  # 0=sagittal, 1=coronal, 2=axial
COMPARE_CLASSES = ["CN", "AD"]

for cls in COMPARE_CLASSES:
    assert cls in class_examples, (
        f"Class '{cls}' not found in class_examples. " f"Available: {list(class_examples.keys())}"
    )

# ── Code selection ──────────────────────────────────────────────
try:
    OLD_CODE  # noqa: F821
    NEW_CODE  # noqa: F821
    print(f"Using upstream-selected OLD_CODE={OLD_CODE}, NEW_CODE={NEW_CODE} " f"(class-discriminative)")
except NameError:
    cn_ids = class_examples["CN"]["id_outputs"][TARGET_LEVEL]
    unique_codes, counts = cn_ids.unique(return_counts=True)
    OLD_CODE = unique_codes[counts.argmax()].item()
    NEW_CODE = unique_codes[counts.argmin()].item()
    if NEW_CODE == OLD_CODE:
        all_present = set(unique_codes.cpu().tolist())
        absent = [c for c in range(nb_entries) if c not in all_present]
        NEW_CODE = absent[0] if absent else (OLD_CODE + 1) % nb_entries
    print(f"Heuristic single-subject pick: OLD_CODE={OLD_CODE}, NEW_CODE={NEW_CODE}")

print(f"Target level: {TARGET_LEVEL} (0=finest, {nb_levels-1}=coarsest)")
print(f"Replacing code {OLD_CODE} -> {NEW_CODE}")
print(f"Comparing classes: {COMPARE_CLASSES}\n")

# ── Embedding diagnostics ───────────────────────────────────────
codebook = vqvae_module.codebooks[TARGET_LEVEL]
emb_old = codebook.embed[:, OLD_CODE]
emb_new = codebook.embed[:, NEW_CODE]
l2_dist = (emb_old - emb_new).norm().item()
cos_sim = F.cosine_similarity(emb_old.unsqueeze(0), emb_new.unsqueeze(0)).item()
print(f"Embedding comparison (code {OLD_CODE} vs {NEW_CODE}):")
print(f"  L2 distance:       {l2_dist:.4f}")
print(f"  Cosine similarity: {cos_sim:.4f}")

all_embeds = codebook.embed.T
sample_idx = torch.randint(0, all_embeds.shape[0], (min(200, nb_entries),))
sample_embeds = all_embeds[sample_idx]
pairwise = torch.cdist(sample_embeds.unsqueeze(0), sample_embeds.unsqueeze(0)).squeeze()
tri_mask = torch.triu(torch.ones_like(pairwise, dtype=torch.bool), diagonal=1)
print(f"  Pairwise L2 stats (sample): mean={pairwise[tri_mask].mean():.4f}, " f"std={pairwise[tri_mask].std():.4f}\n")

# ── Run replacement for each class ──────────────────────────────
class_results = {}  # class_name -> dict with recon_orig, recon_modified, diff, n_replaced, image

for cls in COMPARE_CLASSES:
    ex = class_examples[cls]
    ids = ex["id_outputs"]
    img = ex["image"]
    subj = ex["item"].get("subject", "?")

    print(f"--- {cls} (subject {subj}) ---")
    print(f"  Code map shape at level {TARGET_LEVEL}: {tuple(ids[TARGET_LEVEL].shape)}")

    # Replace codes
    modified_ids = [i.clone() for i in ids]
    n_replaced = (modified_ids[TARGET_LEVEL] == OLD_CODE).sum().item()
    modified_ids[TARGET_LEVEL][modified_ids[TARGET_LEVEL] == OLD_CODE] = NEW_CODE
    print(f"  Replaced {n_replaced} voxels from code {OLD_CODE} -> {NEW_CODE}")

    with torch.no_grad():
        recon_orig = vqvae_module.decode_codes(*ids)
        recon_modified = vqvae_module.decode_codes(*modified_ids)

    # Interpolate to input spatial size if needed
    input_shape = img.shape[2:]
    if recon_orig.shape[2:] != input_shape:
        recon_orig = F.interpolate(recon_orig, size=input_shape, mode="trilinear", align_corners=False)
    if recon_modified.shape[2:] != input_shape:
        recon_modified = F.interpolate(recon_modified, size=input_shape, mode="trilinear", align_corners=False)

    # Mask background using the preprocessed (already brain-masked) input so decoder
    # activations outside the brain don't show up in the diff.
    brain_mask = (img != 0).to(recon_orig.dtype)
    recon_orig = recon_orig * brain_mask
    recon_modified = recon_modified * brain_mask

    diff = (recon_modified - recon_orig).abs()
    print(f"  Diff — max={diff.max().item():.6f}, mean={diff.mean().item():.6f}")
    rel_max = diff.max().item() / (recon_orig.abs().max().item() + 1e-8)
    print(f"  Relative max diff: {rel_max:.4%}")

    class_results[cls] = {
        "recon_orig": recon_orig,
        "recon_modified": recon_modified,
        "diff": diff,
        "n_replaced": n_replaced,
        "image": img,
        "subject": subj,
    }


# ── Visualize side by side ──────────────────────────────────────
def get_slices(vol, axis):
    """Get quarter, mid, and three-quarter slices from (C, D, H, W)."""
    n = vol.shape[axis + 1]
    return [
        vol[0].select(axis, n // 4).cpu().numpy(),
        vol[0].select(axis, n // 2).cpu().numpy(),
        vol[0].select(axis, 3 * n // 4).cpu().numpy(),
    ]


n_classes = len(COMPARE_CLASSES)
# Layout: rows = slice positions (Q1, Mid, Q3)
# For each class: Original | Recon | Modified | Diff  → 4 cols per class
n_cols = 4 * n_classes
slice_labels = ["Q1", "Mid", "Q3"]

# Compute shared intensity ranges across classes for consistent display
all_recons = [class_results[c]["recon_orig"] for c in COMPARE_CLASSES] + [
    class_results[c]["recon_modified"] for c in COMPARE_CLASSES
]
vmin_r = min(r.min().item() for r in all_recons)
vmax_r = max(r.max().item() for r in all_recons)
diff_max = max(max(class_results[c]["diff"].max().item(), 1e-8) for c in COMPARE_CLASSES)

fig, axes = plt.subplots(3, n_cols, figsize=(5 * n_cols, 14))

for ci, cls in enumerate(COMPARE_CLASSES):
    r = class_results[cls]
    col_offset = ci * 4
    volumes = [r["image"], r["recon_orig"], r["recon_modified"], r["diff"]]
    titles = [
        f"{cls} (subj {r['subject']})\nOriginal input",
        f"{cls}\nDecoded (orig codes)",
        f"{cls}\nDecoded (modified)",
        f"{cls}\nDifference",
    ]

    for col_i, (vol, title) in enumerate(zip(volumes, titles)):
        slices = get_slices(vol[0], SLICE_AXIS)
        for row in range(3):
            ax = axes[row, col_offset + col_i]
            if "difference" in title.lower():
                im = ax.imshow(slices[row], cmap="hot", origin="lower", vmin=0, vmax=diff_max)
                if row == 0:
                    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            elif "input" in title.lower():
                ax.imshow(slices[row], cmap="gray", origin="lower")
            else:
                ax.imshow(slices[row], cmap="gray", origin="lower", vmin=vmin_r, vmax=vmax_r)
            if row == 0:
                ax.set_title(title, fontsize=10)
            ax.set_ylabel(slice_labels[row], fontsize=9) if col_i == 0 else None
            ax.axis("off")

fig.suptitle(
    f"Code Replacement — Level {TARGET_LEVEL}, code {OLD_CODE} → {NEW_CODE}\n"
    f"L2 dist={l2_dist:.4f}, cos_sim={cos_sim:.4f}, "
    f"shared diff scale max={diff_max:.6f}",
    fontsize=13,
)
plt.tight_layout()
plt.savefig("codebook_replacement_cn_vs_ad.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Summary table ───────────────────────────────────────────────
print("\nSummary:")
print(f"{'Class':<8} {'Subject':<12} {'# replaced':>12} {'Max diff':>12} {'Mean diff':>12} {'Rel max':>12}")
print("-" * 72)
for cls in COMPARE_CLASSES:
    r = class_results[cls]
    rel = r["diff"].max().item() / (r["recon_orig"].abs().max().item() + 1e-8)
    print(
        f"{cls:<8} {str(r['subject']):<12} {r['n_replaced']:>12d} "
        f"{r['diff'].max().item():>12.6f} {r['diff'].mean().item():>12.6f} {rel:>11.4%}"
    )

print("\n✓ Saved codebook_replacement_cn_vs_ad.png")

In [ ]:
# ============================================================
# §15b. Aggregate across all subjects + baselines + stats test
# ------------------------------------------------------------
# Apply OLD_CODE→NEW_CODE replacement on every T1 scan, plus three
# baselines (self-swap, random code, nearest-neighbour in embedding).
# Per-subject summary stats → CN vs AD Mann-Whitney U.
# ============================================================
import torch.nn.functional as F  # already imported in §15 but be defensive
from scipy import stats as _scipy_stats

# ── Build baseline NEW_CODEs ────────────────────────────────────
_emb = vqvae_module.codebooks[TARGET_LEVEL].embed.detach()  # (embed_dim, nb_entries)
_d = (_emb - _emb[:, OLD_CODE : OLD_CODE + 1]).pow(2).sum(0)
_d[OLD_CODE] = float("inf")
NN_CODE = int(_d.argmin().item())

_rng = np.random.default_rng(0)
RAND_CODE = int(_rng.choice([c for c in range(nb_entries) if c != OLD_CODE]))

SWAPS = {
    "AD-enriched": (OLD_CODE, NEW_CODE),
    "Nearest-nbr": (OLD_CODE, NN_CODE),
    "Random": (OLD_CODE, RAND_CODE),
    "Self (X→X)": (OLD_CODE, OLD_CODE),
}
print(f"Swaps at level {TARGET_LEVEL} (from OLD_CODE={OLD_CODE}):")
for name, (a, b) in SWAPS.items():
    print(f"  {name:<14} → {b}")

# ── Per-subject loop (T1 only) ───────────────────────────────────
records = []
print(f"\nRunning {len(items)} subjects × {len(SWAPS)} swaps ...")
for idx, item in enumerate(items):
    if idx % 25 == 0:
        print(f"  {idx}/{len(items)} ...")
    transformed = val_transforms({"image_t1": item["image"], "image_t2": item["z_image"]})
    img = transformed["image_t1"].unsqueeze(0).to(DEVICE)
    bm = (img != 0).to(img.dtype)

    with torch.no_grad():
        _, _, _, _, _, id_outputs_raw, _, _ = vqvae_model(img, return_recon=True, pool_only=False, view_idx=0)
        id_outputs = id_outputs_raw[::-1]  # finest-first

        recon_orig = vqvae_module.decode_codes(*id_outputs)
        if recon_orig.shape[2:] != img.shape[2:]:
            recon_orig = F.interpolate(recon_orig, size=img.shape[2:], mode="trilinear", align_corners=False)
        recon_orig = recon_orig * bm

        n_at_old = int((id_outputs[TARGET_LEVEL] == OLD_CODE).sum().item())

        for swap_name, (a, b) in SWAPS.items():
            mod_ids = [t.clone() for t in id_outputs]
            mod_ids[TARGET_LEVEL][mod_ids[TARGET_LEVEL] == a] = b
            recon_mod = vqvae_module.decode_codes(*mod_ids)
            if recon_mod.shape[2:] != img.shape[2:]:
                recon_mod = F.interpolate(recon_mod, size=img.shape[2:], mode="trilinear", align_corners=False)
            recon_mod = recon_mod * bm
            d = (recon_mod - recon_orig).abs()
            records.append(
                {
                    "subject": item.get("subject", idx),
                    "label": label_names[item["label"]],
                    "swap": swap_name,
                    "n_at_old": n_at_old,
                    "max_diff": float(d.max().item()),
                    "mean_diff": float(d.mean().item()),
                }
            )

df_records = pd.DataFrame(records)
print(f"\nCollected {len(df_records)} (subject × swap) measurements")

# ── CN vs AD Mann-Whitney per swap × metric ─────────────────────
print(f"\n{'Swap':<14} {'metric':<10} {'CN μ':>11} {'AD μ':>11} " f"{'n_CN':>5} {'n_AD':>5} {'U':>10} {'p':>10}")
print("-" * 82)
for swap_name in SWAPS:
    sub = df_records[df_records["swap"] == swap_name]
    cn = sub[sub["label"] == "CN"]
    ad = sub[sub["label"] == "AD"]
    for metric in ("max_diff", "mean_diff"):
        try:
            U, p = _scipy_stats.mannwhitneyu(cn[metric], ad[metric], alternative="two-sided")
        except ValueError:
            U, p = float("nan"), float("nan")
        print(
            f"{swap_name:<14} {metric:<10} {cn[metric].mean():>11.3e} "
            f"{ad[metric].mean():>11.3e} {len(cn):>5d} {len(ad):>5d} "
            f"{U:>10.1f} {p:>10.2e}"
        )

# ── Plot mean_diff by class for each swap ────────────────────────
fig, axes = plt.subplots(1, len(SWAPS), figsize=(4.2 * len(SWAPS), 4), sharey=False)
if len(SWAPS) == 1:
    axes = [axes]
for ax, swap_name in zip(axes, SWAPS):
    sub = df_records[df_records["swap"] == swap_name]
    data = [sub[sub["label"] == cls]["mean_diff"].values for cls in COMPARE_CLASSES]
    ax.boxplot(data, showfliers=False)
    ax.set_xticks(range(1, len(COMPARE_CLASSES) + 1))
    ax.set_xticklabels(COMPARE_CLASSES)
    for i, cls in enumerate(COMPARE_CLASSES):
        y = sub[sub["label"] == cls]["mean_diff"].values
        x = np.full_like(y, i + 1, dtype=float) + np.random.default_rng(i).uniform(-0.08, 0.08, size=len(y))
        ax.scatter(x, y, s=10, alpha=0.5)
    ax.set_title(f"{swap_name}\n(level {TARGET_LEVEL})", fontsize=10)
    if ax is axes[0]:
        ax.set_ylabel("mean |Δrecon| over brain")
plt.tight_layout()
plt.savefig("codebook_replacement_aggregate.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved codebook_replacement_aggregate.png")

## 16. Save Reconstructions as NIfTI — CN vs AD

Save the original input, VQ-VAE reconstruction, code-modified reconstruction, and
absolute difference map as NIfTI files (`.nii.gz`) for **each class** (CN, AD).
The affine matrix is taken from MONAI's post-transform MetaTensor so the saved
volumes are in the correct preprocessed coordinate space (RAS orientation, resampled,
padded/cropped) and can be loaded directly in FSLeyes, ITK-SNAP, or 3D Slicer.

In [ ]:
import nibabel as nib

# ── Output directory ─────────────────────────────────────────────
NIFTI_OUT_DIR = "nifti_outputs"
os.makedirs(NIFTI_OUT_DIR, exist_ok=True)


# ── Helper: build fallback affine from preprocessing params ──────
def _make_fallback_affine():
    """RAS-aligned affine with the correct spacing, centred at origin."""
    D, H, W = spatial_size
    aff = np.eye(4)
    aff[0, 0] = spacing
    aff[1, 1] = spacing
    aff[2, 2] = spacing
    aff[0, 3] = -spacing * (D - 1) / 2.0
    aff[1, 3] = -spacing * (H - 1) / 2.0
    aff[2, 3] = -spacing * (W - 1) / 2.0
    return aff


# ── Helper: save tensor → .nii.gz ───────────────────────────────
def save_nifti(tensor, affine, filepath, description=""):
    """Save a torch tensor as a compressed NIfTI file.

    Args:
        tensor: (1, 1, D, H, W) or (1, D, H, W) or (D, H, W) torch tensor.
        affine: 4x4 numpy affine matrix.
        filepath: Output path (should end in .nii.gz).
        description: Optional description for the NIfTI header.
    """
    vol = tensor.detach().cpu().float().numpy()
    while vol.ndim > 3:
        vol = vol[0]
    img = nib.Nifti1Image(vol, affine)
    if description:
        img.header["descrip"] = description.encode("ascii", errors="ignore")[:80]
    nib.save(img, filepath)
    print(f"  Saved: {filepath}  {vol.shape}  [{vol.min():.4f}, {vol.max():.4f}]")


# ── Save volumes for each class ──────────────────────────────────
for cls in COMPARE_CLASSES:
    ex = class_examples[cls]
    r = class_results[cls]
    subject_id = ex["item"].get("subject", "unknown")

    # Use MetaTensor affine if available, else fallback
    affine = ex["affine"].copy() if ex["affine"] is not None else _make_fallback_affine()
    affine_src = "MetaTensor" if ex["affine"] is not None else "constructed"

    print(f"\n{'='*60}")
    print(f"{cls} — subject {subject_id}  (affine: {affine_src})")
    print(f"{'='*60}")

    prefix = f"{NIFTI_OUT_DIR}/{cls}_subj_{subject_id}"

    save_nifti(
        r["image"],
        affine,
        f"{prefix}_input.nii.gz",
        description=f"{cls} VQ-VAE input (preprocessed T1)",
    )

    save_nifti(
        r["recon_orig"],
        affine,
        f"{prefix}_recon_orig.nii.gz",
        description=f"{cls} VQ-VAE reconstruction (original codes)",
    )

    save_nifti(
        r["recon_modified"],
        affine,
        f"{prefix}_recon_modified_L{TARGET_LEVEL}_c{OLD_CODE}to{NEW_CODE}.nii.gz",
        description=f"{cls} recon (L{TARGET_LEVEL}: {OLD_CODE}->{NEW_CODE})",
    )

    save_nifti(
        r["diff"],
        affine,
        f"{prefix}_diff_L{TARGET_LEVEL}_c{OLD_CODE}to{NEW_CODE}.nii.gz",
        description=f"{cls} diff (L{TARGET_LEVEL}: {OLD_CODE}->{NEW_CODE})",
    )

print(f"\n✓ All NIfTI volumes saved to {NIFTI_OUT_DIR}/")
print("  Open in FSLeyes / ITK-SNAP / 3D Slicer for interactive inspection.")
print(f"  Affine matches preprocessed space (RAS, {spacing}mm, {spatial_size}).")
print(f"\n  Files per class:")
for cls in COMPARE_CLASSES:
    subj = class_examples[cls]["item"].get("subject", "unknown")
    print(f"    {cls} (subj {subj}): *input, *recon_orig, *recon_modified, *diff")